# 3D Multimodal Brain Tumor Segmentation (MONAI)

**Credit**: This is a port of the [original](https://keras.io/examples/vision/3D_image_classification/) `medicai` (Keras) tutorial to [MONAI](https://monai.io/), the PyTorch-based framework for medical imaging. The data, the task, the model family (`SwinUNETR`), and the evaluation protocol are unchanged, only the framework differs.

Brain tumor segmentation is a core task in medical image analysis, where the goal is to automatically identify and label different tumor sub-regions from 3D MRI scans. Accurate segmentation helps clinicians with diagnosis, treatment planning, and disease monitoring. In this tutorial, we focus on multimodal MRI-based brain tumor segmentation using the widely adopted **BraTS** (**Brain Tumor Segmentation**) dataset.

## The BraTS Dataset

The **BraTS** dataset provides multimodal 3D brain MRI scans, released as NIfTI files (`.nii.gz`). For each patient, four MRI modalities are available:

- **T1** – native T1-weighted MRI
- **T1Gd** – post-contrast T1-weighted MRI
- **T2** – T2-weighted MRI
- **T2-FLAIR** – Fluid Attenuated Inversion Recovery MRI

These scans are collected using different scanners and clinical protocols from 19 institutions, making the dataset diverse and realistic. More details about the dataset can be found in the official [BraTS documentation](https://www.med.upenn.edu/cbica/brats2020/data.html).

## Segmentation Labels

Each scan is manually annotated by **one to four expert raters**, following a standardized annotation protocol and reviewed by experienced neuroradiologists. The segmentation masks contain the following tumor sub-regions:

- **NCR / NET (label 1)** – Necrotic and non-enhancing tumor core
- **ED (label 2)** – Peritumoral edema
- **ET (label 4)** – GD-enhancing tumor
- **0** – Background (non-tumor tissue)

The data are released after preprocessing:

- All modalities are **co-registered**
- Resampled to `1 mm³` isotropic resolution
- **Skull-stripped** for consistency

## Dataset Format and TFRecord Conversion

The original BraTS scans are provided in `.nii` format and can be accessed from Kaggle [here](https://www.kaggle.com/datasets/awsaf49/brats20-dataset-training-validation/). To enable **efficient training pipelines**, the NIfTI files were converted into **TFRecord** format:

- The conversion process is documented [here](https://www.kaggle.com/code/ipythonx/brats-nii-to-tfrecord)
- The preprocessed TFRecord dataset is available [here](https://www.kaggle.com/datasets/ipythonx/brats2020)
- Each TFRecord file contains **up to 20 cases**

Since BraTS does not provide publicly available ground-truth labels for validation or test sets, we will **hold out a subset of TFRecord files** from training for validation purposes.

TFRecord is a TensorFlow container format, so we keep a **CPU-only TensorFlow** import purely to read the shards, then hand the decoded volumes to MONAI/PyTorch. Everything downstream (transforms, model, loss, metrics, inference) is MONAI.

# What This Tutorial Covers

In this tutorial, we provide a step-by-step, end-to-end workflow for brain tumor segmentation using [MONAI](https://github.com/Project-MONAI/MONAI), the PyTorch-based framework for medical imaging. We will walk through:

1. **Loading the Dataset**
    - Read TFRecord files that contain `image`, `label`, and `affine` matrix information.
    - Wrap them in a PyTorch `IterableDataset` and a `monai.data.DataLoader`.
2. **Medical Image Preprocessing**
    - Apply dictionary transforms from `monai.transforms` to prepare the data for model input.
3. **Model Building**
    - Construct a 3D segmentation model with [`SwinUNETR`](https://arxiv.org/abs/2201.01266). You can also experiment with other 3D architectures in `monai.networks.nets`, including [`UNETR`](https://arxiv.org/abs/2103.10504), `SegResNet`, and `UNet`.
4. **Loss and Metrics Definition**
    - Using Dice-based loss functions and segmentation metrics tailored for medical imaging.
5. **Model Evaluation**
    - Performing inference on large 3D volumes using **sliding window inference**
    - Computing per-class evaluation metrics
6. **Visualization of Results**
    - Visualizing predicted segmentation masks for qualitative analysis

By the end of this tutorial, you will have a complete brain tumor segmentation pipeline, from data loading and preprocessing to model training, evaluation, and visualization using modern 3D deep learning techniques and the `monai` framework.

## Installation

We will install [`monai`](https://github.com/Project-MONAI/MONAI) for medical imaging (3D transforms, model
architectures, losses, metrics, sliding-window inference), [`kagglehub`](https://github.com/Kaggle/kagglehub)
for downloading the dataset, and `einops`, which `SwinUNETR` requires.

```shell
!pip install "monai[einops,nibabel,tqdm]" -qU
!pip install kagglehub -qU
```

`tensorflow` is used only to read the TFRecord shards. It is preinstalled on Kaggle/Colab; otherwise:

```shell
!pip install tensorflow-cpu -qU
```

In [ ]:
!pip uninstall -y tensorflow tensorflow-cpu
!pip install --upgrade protobuf -q
!pip install tensorflow-cpu -qU
!pip install "monai[einops,nibabel,tqdm]" -qU
!pip install kagglehub -qU

import os
import warnings
import shutil
import kagglehub
from IPython.display import clear_output
import glob
import numpy as np
import pandas as pd
import torch
import tensorflow as tf
from matplotlib import pyplot as plt
import matplotlib.animation as animation
from matplotlib.colors import ListedColormap
import monai
from monai.data import DataLoader, MetaTensor, decollate_batch
from monai.inferers import sliding_window_inference
from monai.losses import DiceLoss
from monai.metrics import DiceMetric
from monai.networks.nets import SwinUNETR
from monai.transforms import (
    Activations,
    AsDiscrete,
    Compose,
    ConvertToMultiChannelBasedOnBratsClassesd,
    CropForegroundd,
    EnsureTyped,
    NormalizeIntensityd,
    RandFlipd,
    RandShiftIntensityd,
    RandSpatialCropd,
)
from monai.utils import set_determinism

warnings.filterwarnings("ignore")

# TensorFlow is used *only* to read the TFRecord shards. Hide the GPU from it so it
# does not reserve device memory that PyTorch needs.
tf.config.set_visible_devices([], "GPU")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.0/274.0 MB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 158.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 175.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 51.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 142.3 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


Download the dataset from kaggle.

In [ ]:
dataset_id = "ipythonx/brats2020"
destination_path = "brats2020_subset"
os.makedirs(destination_path, exist_ok=True)

Download the 3 shards: 0 and 1st for training set, 36th for validation set.

In [ ]:
for i in [0, 1, 36]:
    filename = f"training_shard_{i}.tfrec"
    print(f"Downloading {filename}...")
    path = kagglehub.dataset_download(dataset_id, path=filename)

    # If the destination file already exists, remove it before moving
    destination_file_path = os.path.join(destination_path, filename)
    if os.path.exists(destination_file_path):
        os.remove(destination_file_path)

    shutil.move(path, destination_path)

Comment this line to keep the logs visible

## Reproducibility

In [ ]:
set_determinism(seed=420)
torch.backends.cudnn.benchmark = True

# Create Multi-label Brain Tumor Labels

The BraTS segmentation task involves multiple tumor sub-regions, and it is formulated as a multi-label segmentation problem. The label combinations are used to define the following clinical regions of interest:

```shell
- Tumor Core (TC): label = 1 or 4
- Whole Tumor (WT): label = 1 or 2 or 4
- Enhancing Tumor (ET): label = 4
```

These region-wise groupings allow for evaluation across different tumor structures relevant for clinical assessment and treatment planning. A sample view is shown below, figure taken from [BraTS-benchmark](https://arxiv.org/abs/2107.02314) paper.

![](https://i.imgur.com/Agnwpxm.png)

MONAI ships this exact conversion as a built-in transform, so no custom class is needed:
[`ConvertToMultiChannelBasedOnBratsClassesd`](https://docs.monai.io/en/stable/transforms.html#converttomultichannelbasedonbratsclassesd)
maps a single-channel label volume with values `{0, 1, 2, 4}` into a 3-channel binary volume ordered
`(TC, WT, ET)` — the same channel order the original notebook built by hand.

## Managing Data and Metadata with `MetaTensor`

The `medicai` version of this tutorial used a `TensorBundle` to carry tensors plus metadata (affine,
spacing, original shape) through the pipeline. MONAI's equivalent is the
[`MetaTensor`](https://docs.monai.io/en/stable/data.html#metatensor): a `torch.Tensor` subclass that
carries an `affine` matrix and a `meta` dictionary along with the array, and that spatial transforms
update automatically as they crop, flip, or resample the data.

MONAI's dictionary transforms (the ones with a `d` suffix) then operate on `key: value` pairs:

```shell
data = {"image": image, "label": label}
```

## Channel-first layout

One important convention difference: `medicai` expects `(depth, height, width, channel)`, while
**MONAI expects channel-first** `(channel, depth, height, width)`. Spatial arguments such as
`spatial_axis` in `RandFlipd` are indexed over the *spatial* dimensions only, i.e. after the channel
axis. We handle the transpose once, when converting a decoded TFRecord into a MONAI sample.

In [ ]:
num_classes = 3  # TC, WT, ET
in_channels = 4  # flair, t1, t1ce, t2
epochs = 50
roi_size = (96, 96, 96)  # random-crop patch size and sliding-window size
sw_batch_size = 4
val_interval = 2

## Transformation

The training pipeline mirrors the original one transform for transform: build the multi-label target,
crop to the brain, take a random `96³` patch, random-flip along each axis, z-score normalize each
modality over its non-zero voxels, and randomly shift intensity.

In [ ]:
train_transform = Compose(
    [
        ConvertToMultiChannelBasedOnBratsClassesd(keys="label"),
        CropForegroundd(
            keys=["image", "label"],
            source_key="image",
            k_divisible=roi_size,
            allow_smaller=True,
        ),
        RandSpatialCropd(keys=["image", "label"], roi_size=roi_size, random_size=False),
        RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=0),
        RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=1),
        RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=2),
        NormalizeIntensityd(keys="image", nonzero=True, channel_wise=True),
        RandShiftIntensityd(keys="image", offsets=0.10, prob=1.0),
        EnsureTyped(keys=["image", "label"], dtype=torch.float32),
    ]
)

Validation keeps the full-size volume: only the label conversion and intensity normalization are
applied, and sliding-window inference handles the size at evaluation time.

In [ ]:
val_transform = Compose(
    [
        ConvertToMultiChannelBasedOnBratsClassesd(keys="label"),
        NormalizeIntensityd(keys="image", nonzero=True, channel_wise=True),
        EnsureTyped(keys=["image", "label"], dtype=torch.float32),
    ]
)

## The `tfrecord` parser

Unchanged from the original notebook: decode the raw bytes for the four modalities plus the label,
reshape them, and stack the modalities into a channel axis.

In [ ]:
def parse_tfrecord_fn(example_proto):
    keys = ["flair", "t1", "t1ce", "t2", "label"]

    feature_description = {}
    for key in keys:
        feature_description[f"{key}_raw"] = tf.io.FixedLenFeature([], tf.string)
        feature_description[f"{key}_shape"] = tf.io.FixedLenFeature([3], tf.int64)
        feature_description[f"{key}_affine"] = tf.io.FixedLenFeature([16], tf.float32)
        feature_description[f"{key}_pixdim"] = tf.io.FixedLenFeature([8], tf.float32)
        feature_description[f"{key}_filename"] = tf.io.FixedLenFeature([], tf.string)

    example = tf.io.parse_single_example(example_proto, feature_description)

    # Decode each modality, reshape to its original dimensions, add a channel axis
    volumes = []
    for key in ["flair", "t1", "t1ce", "t2"]:
        volume = tf.io.decode_raw(example[f"{key}_raw"], tf.float32)
        volume = tf.reshape(volume, example[f"{key}_shape"])
        volumes.append(volume[..., None])

    image = tf.concat(volumes, axis=-1)  # (W, H, D, 4)
    label = tf.io.decode_raw(example["label_raw"], tf.float32)
    label = tf.reshape(label, example["label_shape"])  # (W, H, D)

    return {
        "image": image,
        "label": label,
        "affine": tf.reshape(example["flair_affine"], (4, 4)),  # same for all modalities
    }

## From TFRecord to a MONAI sample

Two things happen here. First the axis order: the stored volumes are `(width, height, depth)` and
MONAI wants channel-first `(channel, depth, height, width)`, so we transpose and reorder the affine
columns to match. Second, the arrays are wrapped in `MetaTensor` so the affine travels with the data
through the transform pipeline.

In [ ]:
def to_monai_sample(image, label, affine):
    """Convert a decoded TFRecord (numpy) into a channel-first MONAI sample."""
    # (W, H, D, C) -> (C, D, H, W) and (W, H, D) -> (D, H, W)
    image = np.ascontiguousarray(np.transpose(image, (3, 2, 1, 0)))
    label = np.ascontiguousarray(np.transpose(label, (2, 1, 0)))

    # reorder the affine's direction columns to match the (D, H, W) axis order
    affine = np.concatenate([affine[:, [2, 1, 0]], affine[:, 3:]], axis=1)
    affine = torch.as_tensor(affine, dtype=torch.float64)

    return {
        "image": MetaTensor(torch.from_numpy(image), affine=affine),
        "label": MetaTensor(torch.from_numpy(label), affine=affine),
    }

## Dataloader

TFRecord files are read sequentially, so we expose them through a PyTorch `IterableDataset`. Shuffling
is delegated to `tf.data`'s shuffle buffer, exactly as in the original pipeline.

We keep `num_workers=0`: each worker would otherwise open its own copy of the TFRecord stream and
replay the same samples. If you want multi-worker loading with `CacheDataset` and random access,
decode the shards to `.npy`/`.nii.gz` once and switch to a regular `monai.data.Dataset`.

In [ ]:
class BraTSTFRecordDataset(torch.utils.data.IterableDataset):
    """Streams BraTS cases out of TFRecord shards and applies a MONAI transform."""

    def __init__(self, tfrecord_datalist, transform, shuffle=False, shuffle_buffer=32):
        super().__init__()
        self.tfrecord_datalist = list(tfrecord_datalist)
        self.transform = transform
        self.shuffle = shuffle
        self.shuffle_buffer = shuffle_buffer

    def _tf_dataset(self):
        dataset = tf.data.TFRecordDataset(self.tfrecord_datalist)
        if self.shuffle:
            dataset = dataset.shuffle(self.shuffle_buffer)
        dataset = dataset.map(parse_tfrecord_fn, num_parallel_calls=tf.data.AUTOTUNE)
        return dataset.prefetch(tf.data.AUTOTUNE)

    def __iter__(self):
        for record in self._tf_dataset():
            sample = to_monai_sample(
                record["image"].numpy(),
                record["label"].numpy(),
                record["affine"].numpy(),
            )
            yield self.transform(sample)

In [ ]:
def train_dataloader(tfrecord_datalist, batch_size=1, shuffle_buffer=32):
    dataset = BraTSTFRecordDataset(
        tfrecord_datalist, train_transform, shuffle=True, shuffle_buffer=shuffle_buffer
    )
    return DataLoader(dataset, batch_size=batch_size, num_workers=0, pin_memory=use_amp)


def val_dataloader(tfrecord_datalist, batch_size=1):
    dataset = BraTSTFRecordDataset(tfrecord_datalist, val_transform, shuffle=False)
    return DataLoader(dataset, batch_size=batch_size, num_workers=0, pin_memory=use_amp)

The training batch size can be set to more than 1 depending on the environment and available resources. However, we intentionally keep the validation batch size as 1 to handle variable-sized samples more flexibly. While padded or ragged batches are alternative options, a batch size of 1 ensures simplicity and consistency during evaluation, especially for 3D medical data.

In [ ]:
tfrecord_pattern = "brats2020_subset/training_shard_*.tfrec"
datalist = sorted(
    glob.glob(tfrecord_pattern),
    key=lambda x: int(x.split("_")[-1].split(".")[0]),
)

In [ ]:
train_datalist = datalist[:-1]
val_datalist = datalist[-1:]
print(len(train_datalist), len(val_datalist))

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = torch.cuda.is_available()

train_ds = train_dataloader(train_datalist, batch_size=1)
val_ds = val_dataloader(val_datalist, batch_size=1)

**sanity check**: Fetch a single validation sample to inspect its shape and values.

In [ ]:
def to_numpy(x):
    """MetaTensor / tensor -> plain numpy array."""
    return np.asarray(x.detach().cpu())

In [ ]:
val_batch = next(iter(val_ds))
test_image = to_numpy(val_batch["image"][0])  # (C, D, H, W)
test_mask = to_numpy(val_batch["label"][0])  # (3, D, H, W)
print(test_image.shape, test_mask.shape, np.unique(test_mask))
print(test_image.min(), test_image.max())

**sanity check**: Visualize the middle slice of the image and its corresponding label.

In [ ]:
slice_no = test_image.shape[1] // 2

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))
ax1.imshow(test_image[0, slice_no], cmap="gray")
ax1.set_title(f"Image shape: {test_image.shape}")
ax2.imshow(test_mask[:, slice_no].transpose(1, 2, 0))
ax2.set_title(f"Label shape: {test_mask.shape}")
plt.show()

**sanity check**: Visualize sample image and label channels at middle slice index.

In [ ]:
print(f"image shape: {test_image.shape}")
plt.figure("image", (24, 6))
for i in range(in_channels):
    plt.subplot(1, 4, i + 1)
    plt.title(f"image channel {i}")
    plt.imshow(test_image[i, slice_no], cmap="gray")
plt.show()

In [ ]:
print(f"label shape: {test_mask.shape}")
plt.figure("label", (18, 6))
for i in range(num_classes):
    plt.subplot(1, 3, i + 1)
    plt.title(f"label channel {i}")
    plt.imshow(test_mask[i, slice_no])
plt.show()

## Model

We will be using the 3D model architecture Swin UNEt TRansformers, i.e., [`SwinUNETR`](https://arxiv.org/abs/2201.01266). It was used in the BraTS 2021 segmentation challenge by NVIDIA. The model was among the top-performing methods. It uses a Swin Transformer encoder to extract features at five different resolutions. A CNN-based decoder is connected to each resolution using skip connections.

The BraTS dataset provides four input modalities: `flair`, `t1`, `t1ce`, and `t2` and three multi-label outputs: `tumor-core`, `whole-tumor`, and `enhancing-tumor`. Accordingly, we will initiate the model with `4` input channels and `3` output channels.

![](https://i.imgur.com/OInMRGp.png)

`feature_size=48` is the configuration used in the BraTS 2021 submission. `use_checkpoint=True` trades
some compute for a large reduction in activation memory, which is what makes `96³` patches fit on a
single GPU.

```shell
# # check available architectures
# from monai.networks import nets
# print([n for n in dir(nets) if not n.startswith("_")])
```

In [ ]:
model_kwargs = dict(
    in_channels=in_channels,
    out_channels=num_classes,
    feature_size=48,
    use_checkpoint=True,
)

try:
    # MONAI >= 1.5 infers the spatial size, `img_size` was removed
    model = SwinUNETR(**model_kwargs)
except TypeError:
    # older MONAI releases still require `img_size`
    model = SwinUNETR(img_size=roi_size, **model_kwargs)

model = model.to(device)

In [ ]:
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"{model.__class__.__name__}: {n_params / 1e6:.1f}M trainable parameters")

## Loss and Metrics

The original notebook used `medicai`'s `BinaryDiceCELoss`. MONAI's `DiceCELoss` is *not* a drop-in
replacement here: for multi-channel input it applies softmax cross-entropy across channels, which
assumes mutually exclusive classes. BraTS regions overlap (every ET voxel is also a TC and a WT
voxel), so the cross-entropy term has to be **binary**. We combine `DiceLoss(sigmoid=True)` with
`BCEWithLogitsLoss` explicitly.

In [ ]:
class DiceBCELoss(torch.nn.Module):
    """Dice + binary cross-entropy, for overlapping (multi-label) targets."""

    def __init__(self, lambda_dice=1.0, lambda_bce=1.0):
        super().__init__()
        self.dice = DiceLoss(sigmoid=True, squared_pred=True, smooth_nr=0.0, smooth_dr=1e-5)
        self.bce = torch.nn.BCEWithLogitsLoss()
        self.lambda_dice = lambda_dice
        self.lambda_bce = lambda_bce

    def forward(self, logits, target):
        target = target.float()
        return self.lambda_dice * self.dice(logits, target) + self.lambda_bce * self.bce(
            logits, target
        )

A single `DiceMetric` with `reduction="mean_batch"` returns one Dice score **per channel**, which
replaces the four separate `BinaryDiceMetric` instances of the original notebook: index `0` is TC,
`1` is WT, `2` is ET, and their mean is the overall score. `ignore_empty=True` skips classes that are
absent from the ground truth instead of scoring them as zero.

Predictions come out as logits, so the post-processing chain applies a sigmoid and thresholds at
`0.5` before the metric sees them.

In [ ]:
loss_fn = DiceBCELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
scaler = torch.amp.GradScaler(device.type, enabled=use_amp)

dice_metric = DiceMetric(include_background=True, reduction="mean_batch", ignore_empty=True)
post_pred = Compose([Activations(sigmoid=True), AsDiscrete(threshold=0.5)])

## Sliding window validation

The `medicai` version used a `SlidingWindowInferenceCallback`. In PyTorch the training loop is
explicit, so we call [`sliding_window_inference`](https://docs.monai.io/en/stable/inferers.html#sliding-window-inference)
directly: the full validation volume is covered with overlapping `96³` windows, blended with a
gaussian weighting so that window borders do not produce seams.

In [ ]:
@torch.no_grad()
def run_validation(model, loader, metric, post_pred):
    model.eval()
    metric.reset()

    for batch in loader:
        images = batch["image"].to(device)
        labels = batch["label"].to(device)

        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            logits = sliding_window_inference(
                inputs=images,
                roi_size=roi_size,
                sw_batch_size=sw_batch_size,
                predictor=model,
                overlap=0.5,
                mode="gaussian",
            )

        preds = [post_pred(p) for p in decollate_batch(logits.float())]
        metric(y_pred=preds, y=decollate_batch(labels))

    per_class = metric.aggregate().cpu().numpy()  # (3,) -> TC, WT, ET
    metric.reset()

    return {
        "dice": float(np.nanmean(per_class)),
        "dice_tc": float(per_class[0]),
        "dice_wt": float(per_class[1]),
        "dice_et": float(per_class[2]),
    }

## Training

Set more epochs for better optimization. Validation runs every `val_interval` epochs and the best
checkpoint (by mean Dice) is written to disk.

In [ ]:
xweights_path = "brats.model.pt"
history = {k: [] for k in ["epoch", "loss", "val_dice", "val_dice_tc", "val_dice_wt", "val_dice_et"]}
best_dice = -1.0

for epoch in range(1, epochs + 1):
    model.train()
    epoch_loss, steps = 0.0, 0

    for batch in train_ds:
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            logits = model(images)
            loss = loss_fn(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        epoch_loss += loss.item()
        steps += 1

    epoch_loss /= max(steps, 1)
    history["epoch"].append(epoch)
    history["loss"].append(epoch_loss)
    log = f"epoch {epoch}/{epochs} - loss: {epoch_loss:.4f}"

    if epoch % val_interval == 0:
        scores = run_validation(model, val_ds, dice_metric, post_pred)
        for key, value in scores.items():
            history[f"val_{key}"].append(value)
        log += " - " + " - ".join(f"val_{k}: {v:.4f}" for k, v in scores.items())

        if scores["dice"] > best_dice:
            best_dice = scores["dice"]
            torch.save(model.state_dict(), xweights_path)
            log += f" (saved {xweights_path})"
    else:
        for key in ["val_dice", "val_dice_tc", "val_dice_wt", "val_dice_et"]:
            history[key].append(np.nan)

    print(log)

Comment this line to keep the logs visible

In [ ]:
clear_output()

Let's take a quick look at how our model performed during training. We will first print the available metrics recorded in the training history, save them to a CSV file for future reference, and then visualize them to better understand the model's learning progress over epochs.

In [ ]:
def plot_training_history(history_df):
    metrics = history_df.columns
    n_metrics = len(metrics)
    n_rows = 2
    n_cols = (n_metrics + 1) // 2  # ceiling division for columns

    plt.figure(figsize=(5 * n_cols, 5 * n_rows))
    for idx, metric in enumerate(metrics):
        plt.subplot(n_rows, n_cols, idx + 1)
        plt.plot(history_df[metric], label=metric, marker="o")
        plt.title(metric)
        plt.xlabel("Epoch")
        plt.ylabel("Value")
        plt.grid(True)
        plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
his_csv = pd.DataFrame(history).set_index("epoch")
print(list(his_csv.columns))
his_csv.to_csv("brats.history.csv")
plot_training_history(his_csv)

## Evaluation

The original notebook restored Keras weights (`brats.model.weights.h5`) trained for ~30 epochs on the
full dataset. Those cannot be loaded into a PyTorch model — the parameter layout and naming are
framework specific — so there is no equivalent download here. Two options:

1. Train longer with this notebook and reuse the checkpoint saved above (what the next cell does).
2. Start from a [MONAI Model Zoo](https://monai.io/model-zoo.html) bundle, e.g.
   `python -m monai.bundle download --name brats_mri_segmentation --bundle_dir ./bundles`.
   Note that the BraTS bundle ships a `SegResNet`, so it replaces the model rather than supplying
   weights for `SwinUNETR`.

With only a handful of epochs the scores below will be low; that is expected.

In [ ]:
if os.path.exists(weights_path):
    model.load_state_dict(torch.load(weights_path, map_location=device))
    print(f"loaded weights from {weights_path}")
else:
    print("no checkpoint found, evaluating the in-memory model")

In this section, we perform sliding window inference on the validation dataset and compute Dice scores for overall segmentation quality as well as specific tumor subregions:

 - Tumor Core (TC)
 - Whole Tumor (WT)
 - Enhancing Tumor (ET)

In [ ]:
scores = run_validation(model, val_ds, dice_metric, post_pred)

Comment this line to keep the logs visible

In [ ]:
clear_output()

In [ ]:
print(f"Dice Score: {scores['dice']:.4f}")
print(f"Dice Score on tumor core (TC): {scores['dice_tc']:.4f}")
print(f"Dice Score on whole tumor (WT): {scores['dice_wt']:.4f}")
print(f"Dice Score on enhancing tumor (ET): {scores['dice_et']:.4f}")

## Analyse and Visualize

Let's analyse the model predictions and visualize them. The test transformation is the same as the
validation one.

In [ ]:
test_transform = val_transform

Let's load the `tfrecord` file and check its properties. We keep the decoded numpy arrays around so
we can build the raw (untransformed) sample for display and the preprocessed sample for inference.

In [ ]:
index = 0
raw_dataset = tf.data.TFRecordDataset(val_datalist[index]).map(parse_tfrecord_fn)
record = next(iter(raw_dataset))

raw_image = record["image"].numpy()
raw_label = record["label"].numpy()
raw_affine = record["affine"].numpy()

In [ ]:
sample = to_monai_sample(raw_image, raw_label, raw_affine)
orig_image = to_numpy(sample["image"])  # (4, D, H, W)
orig_label = to_numpy(sample["label"])  # (D, H, W)
print(orig_image.shape, orig_label.shape, np.unique(orig_label))

Run the transformation to prepare the inputs.

In [ ]:
transformed = test_transform(to_monai_sample(raw_image, raw_label, raw_affine))
pre_image, pre_label = transformed["image"], transformed["label"]
print(pre_image.shape, pre_label.shape)

Pass the preprocessed sample to sliding window inference, ensuring that a batch axis is added to the input beforehand.

In [ ]:
model.eval()
with torch.no_grad():
    with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
        y_pred = sliding_window_inference(
            inputs=pre_image[None].to(device),
            roi_size=roi_size,
            sw_batch_size=sw_batch_size,
            predictor=model,
            overlap=0.5,
            mode="gaussian",
        )

Comment this line to keep the logs visible

In [ ]:
clear_output()

In [ ]:
print(y_pred.shape)

After running inference, we remove the batch dimension and apply a `sigmoid` activation to obtain class probabilities. We then threshold the probabilities at `0.5` to generate the final binary segmentation map.

In [ ]:
y_pred_logits = y_pred.float()[0]
y_pred_prob = to_numpy(torch.sigmoid(y_pred_logits))
segment = (y_pred_prob > 0.5).astype(int)  # (3, D, H, W)
print(segment.shape, np.unique(segment))

We compare the ground truth (`pre_label`) and the predicted segmentation (`segment`) for each tumor sub-region. Each sub-plot shows a specific channel corresponding to a tumor type: TC, WT, and ET. Here we visualize the `80th` axial slice across the three channels.

In [ ]:
label_map = {0: "TC", 1: "WT", 2: "ET"}
pre_label_np = to_numpy(pre_label)

In [ ]:
plt.figure(figsize=(16, 4))
for i in range(pre_label_np.shape[0]):
    plt.subplot(1, 3, i + 1)
    plt.title(f"label channel {label_map[i]}")
    plt.imshow(pre_label_np[i, 80])
plt.show()

In [ ]:
plt.figure(figsize=(16, 4))
for i in range(segment.shape[0]):
    plt.subplot(1, 3, i + 1)
    plt.title(f"pred channel {label_map[i]}")
    plt.imshow(segment[i, 80])
plt.show()

The predicted output is a multi-channel binary map, where each channel corresponds to a specific tumor region. To visualize it against the original ground truth, we convert it into a single-channel label map. Here we assign:

  - Label 1 for Tumor Core (TC)
  - Label 2 for Whole Tumor (WT)
  - Label 4 for Enhancing Tumor (ET)

The label values are chosen to match typical conventions used in medical segmentation benchmarks like BraTS.

In [ ]:
prediction = np.zeros(segment.shape[1:], dtype="float32")  # (D, H, W)
prediction[segment[1] == 1] = 2
prediction[segment[0] == 1] = 1
prediction[segment[2] == 1] = 4

In [ ]:
print("label ", orig_label.shape, np.unique(orig_label))
print("predicted ", prediction.shape, np.unique(prediction))

Let's begin by examining the original input slices from the MRI scan. The input contains four channels corresponding to different MRI modalities:

  - FLAIR
  - T1
  - T1CE (T1 with contrast enhancement)
  - T2

We display the same slice number across all modalities for comparison.

In [ ]:
slice_map = {0: "flair", 1: "t1", 2: "t1ce", 3: "t2"}
slice_num = 75

In [ ]:
plt.figure(figsize=(16, 4))
for i in range(orig_image.shape[0]):
    plt.subplot(1, 4, i + 1)
    plt.title(f"Original channel: {slice_map[i]}")
    plt.imshow(orig_image[i, slice_num], cmap="gray")
plt.tight_layout()
plt.show()

Next, we compare this input with the ground truth label and the predicted segmentation on the same slice. This provides visual insight into how well the model has localized and segmented the tumor regions.

In [ ]:
plt.figure("image", (15, 5))

plt.subplot(1, 3, 1)
plt.title("image")
plt.imshow(orig_image[0, slice_num], cmap="gray")

plt.subplot(1, 3, 2)
plt.title("label")
plt.imshow(orig_label[slice_num])

plt.subplot(1, 3, 3)
plt.title("prediction")
plt.imshow(prediction[slice_num])

plt.tight_layout()
plt.show()

Finally, create a clean GIF visualizer showing the input image, ground-truth label, and model prediction.

The input volume contains large black margins, so we crop the foreground region of interest (ROI).
Note that `CropForegroundd` needs channel-first inputs, so the single-channel label and prediction
volumes get a leading axis before cropping.

In [ ]:
crop_foreground = CropForegroundd(
    keys=["image", "label", "prediction"], source_key="image", allow_smaller=True
)

In [ ]:
results = crop_foreground(
    {
        "image": sample["image"],
        "label": torch.as_tensor(orig_label)[None],
        "prediction": torch.as_tensor(prediction)[None],
    }
)
crop_orig_image = to_numpy(results["image"])  # (4, D, H, W)
crop_orig_label = to_numpy(results["label"])[0]  # (D, H, W)
crop_prediction = to_numpy(results["prediction"])[0]  # (D, H, W)
print(crop_orig_image.shape, crop_orig_label.shape, crop_prediction.shape)

Prepare visualization-friendly label maps by remapping label values to a compact index range.

In [ ]:
def to_viz_map(volume):
    viz = np.zeros_like(volume, dtype="uint8")
    viz[volume == 1] = 1
    viz[volume == 2] = 2
    viz[volume == 4] = 3
    return viz


viz_label = to_viz_map(crop_orig_label)
viz_pred = to_viz_map(crop_prediction)

Colormap for background, tumor core, edema, and enhancing regions

In [ ]:
cmap = ListedColormap(
    [
        "#000000",  # background
        "#E57373",  # muted red
        "#64B5F6",  # muted blue
        "#81C784",  # muted green
    ]
)

Create side-by-side views for input, label, and prediction, then animate through the slices.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 4))
ax_img, ax_lbl, ax_pred = axes

img_im = ax_img.imshow(crop_orig_image[0, 0], cmap="gray")
lbl_im = ax_lbl.imshow(viz_label[0], vmin=0, vmax=3, cmap=cmap, interpolation="nearest")
pred_im = ax_pred.imshow(viz_pred[0], vmin=0, vmax=3, cmap=cmap, interpolation="nearest")

# Tight layout for a compact GIF
plt.subplots_adjust(left=0.01, right=0.99, bottom=0.02, top=0.8, wspace=0.01)

for ax, t in zip(axes, ["FLAIR", "Label", "Prediction"]):
    ax.set_title(t, fontsize=19, pad=10)
    ax.axis("off")
    ax.set_adjustable("box")


def update(i):
    img_im.set_data(crop_orig_image[0, i])
    lbl_im.set_data(viz_label[i])
    pred_im.set_data(viz_pred[i])
    fig.suptitle(f"Slice {i}", fontsize=14)
    return img_im, lbl_im, pred_im


ani = animation.FuncAnimation(
    fig, update, frames=crop_orig_image.shape[1], interval=120
)
ani.save("segmentation_slices.gif", writer="pillow", dpi=100)
plt.close(fig)

When you open the saved GIF, you should see a visualization similar to this.

![Animation of the brain tumor segmentation results](https://i.imgur.com/CbaQGf2.gif)

## Additional Resources

- [MONAI BraTS brain tumour segmentation tutorial](https://github.com/Project-MONAI/tutorials/blob/main/3d_segmentation/brats_segmentation_3d.ipynb)
- [MONAI SwinUNETR BraTS pipeline](https://github.com/Project-MONAI/research-contributions/tree/main/SwinUNETR/BRATS21)
- [MONAI Model Zoo](https://monai.io/model-zoo.html)
- [BraTS .nii to TFRecord](https://www.kaggle.com/code/ipythonx/brats-nii-to-tfrecord)
- [BraTS Segmentation on Multi-GPU (Keras/medicai original)](https://www.kaggle.com/code/ipythonx/3d-brats-segmentation-in-keras-multi-gpu)